# Garmin Sleep Analysis

This notebook pulls data from `mart_sleep_correlates_daily` and summarizes the strongest observed relationships in the Garmin sleep dataset.

Focus areas:
- sample coverage for the selected date window,
- missingness across the main analysis columns,
- same-day and lagged correlations with `sleep_score`,
- workout vs non-workout comparisons,
- best and worst nights in the selected range.


## How To Run This Notebook

This notebook reads from mart_sleep_correlates_daily using the same PostgreSQL settings defined in the project .env file.

Before running:
1. Make sure PostgreSQL is running.
2. Make sure mart_sleep_correlates_daily has been built and contains data.
3. Open this notebook in VS Code or Jupyter.
4. Select the project Python interpreter or virtual environment.
5. Run all cells from top to bottom.

The default analysis window is set in the notebook and can be adjusted before rerunning.


## Setup

The notebook uses the same project config as the CLI and Streamlit app, so it reads database connection details from `.env` through `app.utils.config`.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.utils.config import get_settings

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 200)

settings = get_settings()
engine = create_engine(settings.sqlalchemy_url)


## Analysis Window

Adjust the date range here if you want to rerun the notebook for a different slice of the mart.


In [2]:
START_DATE = "2025-02-01"
END_DATE = "2025-06-30"
ONLY_NIGHTS_WITH_HRV = False

NUMERIC_COLUMNS = [
    "sleep_score",
    "sleep_duration_minutes",
    "deep_sleep_minutes",
    "rem_sleep_minutes",
    "light_sleep_minutes",
    "awake_minutes",
    "avg_stress",
    "max_stress",
    "steps",
    "calories_burned",
    "avg_heart_rate",
    "resting_heart_rate",
    "workout_count",
    "activity_minutes",
    "hrv_value",
    "prior_day_steps",
    "prior_day_calories_burned",
    "prior_day_avg_stress",
    "prior_day_resting_heart_rate",
    "avg_overnight_hrv",
    "avg_sleep_stress",
    "total_distance_meters",
    "active_seconds",
    "highly_active_seconds",
    "activity_calories",
    "activity_distance_meters",
    "last_night_5min_high",
    "hrv_weekly_avg",
]

query = text(
    """
    SELECT *
    FROM mart_sleep_correlates_daily
    WHERE sleep_date BETWEEN :start_date AND :end_date
    ORDER BY sleep_date
    """
)

with engine.connect() as conn:
    df = pd.read_sql_query(query, conn, params={"start_date": START_DATE, "end_date": END_DATE})

if df.empty:
    raise ValueError("The selected date range returned no rows from mart_sleep_correlates_daily.")

df["sleep_date"] = pd.to_datetime(df["sleep_date"])
for column in ("sleep_start_gmt", "sleep_end_gmt"):
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

for column in NUMERIC_COLUMNS:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

if ONLY_NIGHTS_WITH_HRV:
    df = df[df["hrv_value"].notna()].copy()

df.head()


,sleep_date,sleep_score,sleep_duration_minutes,deep_sleep_minutes,rem_sleep_minutes,light_sleep_minutes,awake_minutes,avg_stress,max_stress,steps,calories_burned,avg_heart_rate,resting_heart_rate,workout_count,activity_minutes,hrv_value,prior_day_steps,prior_day_calories_burned,prior_day_avg_stress,prior_day_resting_heart_rate,avg_overnight_hrv,avg_sleep_stress,total_distance_meters,active_seconds,highly_active_seconds,stress_qualifier,activity_calories,activity_distance_meters,last_night_5min_high,hrv_weekly_avg,hrv_status,sleep_feedback,sleep_score_insight,sleep_start_gmt,sleep_end_gmt,sleep_source_file,daily_health_source_file,activities_source_file,hrv_source_file
0,2025-02-01,55,278.0,51.0,64.0,163.0,42.0,26.0,99,5623,1927.0,82.5,48,0,0.0,49.0,NaN,NaN,NaN,NaN,NaN,13.0,4453.0,2757,2589,UNKNOWN,0.0,0.0,93.0,51.0,BALANCED,POSITIVE_SHORT_BUT_REFRESHING,NONE,2025-01-31 21:59:00+00:00,2025-02-01 03:32:00+00:00,2025-02-01_20260425T010256Z.json,2025-02-01_20260425T010259Z.json,2025-02-01_to_2025-02-28_20260425T010306Z.json,2025-02-01_20260425T010306Z.json
1,2025-02-02,87,471.0,107.0,119.0,245.0,3.0,29.0,99,4452,1910.0,84.5,50,0,0.0,45.0,5623.0,1927.0,26.0,48.0,NaN,14.0,3546.0,5451,960,BALANCED,0.0,0.0,67.0,51.0,BALANCED,POSITIVE_LONG_AND_REFRESHING,NONE,2025-02-01 18:06:00+00:00,2025-02-02 02:00:00+00:00,2025-02-02_20260425T010256Z.json,2025-02-02_20260425T010259Z.json,2025-02-01_to_2025-02-28_20260425T010306Z.json,2025-02-02_20260425T010306Z.json
2,2025-02-04,69,363.0,111.0,55.0,197.0,9.0,31.0,99,9679,2041.0,93.0,51,0,0.0,44.0,4452.0,1910.0,29.0,50.0,NaN,18.0,7596.0,5525,3400,UNKNOWN,0.0,0.0,72.0,51.0,BALANCED,POSITIVE_SHORT_BUT_DEEP,NONE,2025-02-03 19:40:00+00:00,2025-02-04 01:52:00+00:00,2025-02-04_20260425T010257Z.json,2025-02-04_20260425T010300Z.json,2025-02-01_to_2025-02-28_20260425T010306Z.json,2025-02-04_20260425T010307Z.json
3,2025-02-05,69,407.0,124.0,31.0,252.0,19.0,22.0,90,11206,2217.0,76.0,52,0,0.0,57.0,9679.0,2041.0,31.0,51.0,NaN,15.0,11060.0,3921,3953,UNKNOWN,0.0,0.0,90.0,49.0,BALANCED,NEGATIVE_NOT_ENOUGH_REM,NONE,2025-02-04 19:06:00+00:00,2025-02-05 02:12:00+00:00,2025-02-05_20260425T010257Z.json,2025-02-05_20260425T010300Z.json,2025-02-01_to_2025-02-28_20260425T010306Z.json,2025-02-05_20260425T010307Z.json
4,2025-02-06,59,299.0,130.0,69.0,100.0,11.0,18.0,74,9559,1967.0,70.5,50,0,0.0,46.0,11206.0,2217.0,22.0,52.0,NaN,19.0,7694.0,4535,2987,UNKNOWN,0.0,0.0,72.0,48.0,BALANCED,POSITIVE_SHORT_BUT_DEEP,POSITIVE_LATE_BED_TIME,2025-02-05 19:35:00+00:00,2025-02-06 00:45:00+00:00,2025-02-06_20260425T010257Z.json,2025-02-06_20260425T010300Z.json,2025-02-01_to_2025-02-28_20260425T010306Z.json,2025-02-06_20260425T010307Z.json


## Coverage And Descriptive Statistics

This section confirms how many nights are available in the current slice and summarizes the main sleep and recovery fields.


In [3]:
coverage_summary = pd.DataFrame(
    [
        {"metric": "start_date", "value": START_DATE},
        {"metric": "end_date", "value": END_DATE},
        {"metric": "nights", "value": int(len(df))},
        {"metric": "nights_with_hrv", "value": int(df["hrv_value"].notna().sum())},
        {"metric": "nights_with_avg_heart_rate", "value": int(df["avg_heart_rate"].notna().sum())},
        {"metric": "nights_with_resting_heart_rate", "value": int(df["resting_heart_rate"].notna().sum())},
        {"metric": "workout_nights", "value": int((df["workout_count"].fillna(0) > 0).sum())},
    ]
)
coverage_summary


,metric,value
0,start_date,2025-02-01
1,end_date,2025-06-30
2,nights,140
3,nights_with_hrv,140
4,nights_with_avg_heart_rate,140
5,nights_with_resting_heart_rate,140
6,workout_nights,17


In [4]:
summary_columns = [
    "sleep_score",
    "sleep_duration_minutes",
    "deep_sleep_minutes",
    "rem_sleep_minutes",
    "light_sleep_minutes",
    "awake_minutes",
    "avg_stress",
    "avg_heart_rate",
    "resting_heart_rate",
    "hrv_value",
    "steps",
    "calories_burned",
    "activity_minutes",
]
available_summary_columns = [column for column in summary_columns if column in df.columns]
df[available_summary_columns].describe().round(2).transpose()


,count,mean,std,min,25%,50%,75%,max
sleep_score,140.0,69.51,13.10,32.0,62.00,72.0,79.00,95.00
sleep_duration_minutes,140.0,440.94,88.82,203.0,395.00,455.5,505.00,662.92
deep_sleep_minutes,140.0,88.66,30.54,13.0,68.75,89.0,108.25,195.00
rem_sleep_minutes,140.0,82.10,38.62,0.0,54.75,82.5,107.00,184.00
light_sleep_minutes,140.0,270.22,72.36,96.0,228.75,273.5,321.75,446.00
awake_minutes,140.0,38.66,37.32,0.0,9.00,26.5,56.50,192.00
avg_stress,140.0,35.10,6.46,15.0,31.00,34.5,40.00,48.00
avg_heart_rate,140.0,90.66,10.67,66.5,83.50,89.5,97.12,117.50
resting_heart_rate,140.0,51.78,2.72,45.0,50.00,52.0,54.00,58.00
hrv_value,140.0,46.26,5.22,37.0,43.00,46.0,49.00,66.00


## Missingness Review

Missingness matters here because Garmin endpoints are not uniformly complete across all dates and metrics.


In [5]:
missingness_columns = [
    "sleep_score",
    "sleep_duration_minutes",
    "avg_stress",
    "steps",
    "calories_burned",
    "avg_heart_rate",
    "resting_heart_rate",
    "workout_count",
    "activity_minutes",
    "hrv_value",
    "prior_day_avg_stress",
    "prior_day_resting_heart_rate",
]
missingness = pd.DataFrame(
    {
        "column": missingness_columns,
        "non_null_rows": [int(df[column].notna().sum()) for column in missingness_columns],
        "null_rows": [int(df[column].isna().sum()) for column in missingness_columns],
    }
)
missingness["pct_non_null"] = (missingness["non_null_rows"] / len(df) * 100).round(1)
missingness.sort_values(["pct_non_null", "column"], ascending=[False, True]).reset_index(drop=True)


,column,non_null_rows,null_rows,pct_non_null
0,activity_minutes,140,0,100.0
1,avg_heart_rate,140,0,100.0
2,avg_stress,140,0,100.0
3,calories_burned,140,0,100.0
4,hrv_value,140,0,100.0
5,resting_heart_rate,140,0,100.0
6,sleep_duration_minutes,140,0,100.0
7,sleep_score,140,0,100.0
8,steps,140,0,100.0
9,workout_count,140,0,100.0


## Sleep Score Correlations

This reproduces the core correlation view used in the dashboard and makes it easier to compare same-day and lagged signals.


In [6]:
def build_sleep_score_correlation_summary(frame: pd.DataFrame) -> pd.DataFrame:
    correlation_columns = [
        "avg_stress",
        "steps",
        "calories_burned",
        "avg_heart_rate",
        "resting_heart_rate",
        "workout_count",
        "activity_minutes",
        "hrv_value",
        "prior_day_steps",
        "prior_day_calories_burned",
        "prior_day_avg_stress",
        "prior_day_resting_heart_rate",
    ]
    available_columns = [
        column
        for column in correlation_columns
        if column in frame.columns and frame[column].notna().any()
    ]
    if not available_columns:
        return pd.DataFrame(columns=["metric", "correlation_with_sleep_score"])

    corr_df = (
        frame[["sleep_score", *available_columns]]
        .corr(numeric_only=True)["sleep_score"]
        .drop(labels=["sleep_score"])
        .dropna()
        .sort_values(key=lambda series: series.abs(), ascending=False)
        .rename_axis("metric")
        .reset_index(name="correlation_with_sleep_score")
    )
    return corr_df

sleep_score_corr = build_sleep_score_correlation_summary(df)
sleep_score_corr.round(4)


,metric,correlation_with_sleep_score
0,avg_stress,-0.3669
1,hrv_value,0.3387
2,resting_heart_rate,-0.3203
3,prior_day_avg_stress,-0.2345
4,prior_day_resting_heart_rate,-0.2015
5,steps,-0.1232
6,activity_minutes,-0.1106
7,prior_day_steps,0.1068
8,prior_day_calories_burned,0.0665
9,avg_heart_rate,-0.0573


In [7]:
lagged_vs_same_day = pd.DataFrame(
    [
        {
            "family": "stress",
            "same_day_metric": "avg_stress",
            "same_day_corr": sleep_score_corr.loc[sleep_score_corr["metric"] == "avg_stress", "correlation_with_sleep_score"].squeeze(),
            "lagged_metric": "prior_day_avg_stress",
            "lagged_corr": sleep_score_corr.loc[sleep_score_corr["metric"] == "prior_day_avg_stress", "correlation_with_sleep_score"].squeeze(),
        },
        {
            "family": "resting_heart_rate",
            "same_day_metric": "resting_heart_rate",
            "same_day_corr": sleep_score_corr.loc[sleep_score_corr["metric"] == "resting_heart_rate", "correlation_with_sleep_score"].squeeze(),
            "lagged_metric": "prior_day_resting_heart_rate",
            "lagged_corr": sleep_score_corr.loc[sleep_score_corr["metric"] == "prior_day_resting_heart_rate", "correlation_with_sleep_score"].squeeze(),
        },
    ]
)
lagged_vs_same_day["absolute_gap"] = (
    lagged_vs_same_day["same_day_corr"].abs() - lagged_vs_same_day["lagged_corr"].abs()
).round(4)
lagged_vs_same_day.round(4)


,family,same_day_metric,same_day_corr,lagged_metric,lagged_corr,absolute_gap
0,stress,avg_stress,-0.3669,prior_day_avg_stress,-0.2345,0.1324
1,resting_heart_rate,resting_heart_rate,-0.3203,prior_day_resting_heart_rate,-0.2015,0.1188


## Workout Comparison

This split checks whether nights after days with at least one recorded workout differ from nights without a workout.


In [8]:
df["day_type"] = df["workout_count"].fillna(0).gt(0).map({True: "workout_day", False: "no_workout_day"})
workout_summary = (
    df.groupby("day_type", dropna=False)
    .agg(
        nights=("sleep_date", "count"),
        avg_sleep_score=("sleep_score", "mean"),
        avg_sleep_duration_minutes=("sleep_duration_minutes", "mean"),
        avg_stress=("avg_stress", "mean"),
        avg_hrv=("hrv_value", "mean"),
    )
    .round(2)
    .reset_index()
)
workout_summary


,day_type,nights,avg_sleep_score,avg_sleep_duration_minutes,avg_stress,avg_hrv
0,no_workout_day,123,69.11,444.56,35.21,46.02
1,workout_day,17,72.35,414.76,34.29,48.00


## Standout Nights

The next two tables surface the best and worst nights in the filtered range using sleep score first and sleep duration as a tiebreaker.


In [9]:
standout_columns = [
    "sleep_date",
    "sleep_score",
    "sleep_duration_minutes",
    "avg_stress",
    "resting_heart_rate",
    "hrv_value",
    "steps",
    "workout_count",
]

best_nights = (
    df.sort_values(["sleep_score", "sleep_duration_minutes"], ascending=[False, False])
    .head(5)[standout_columns]
    .copy()
)
worst_nights = (
    df.sort_values(["sleep_score", "sleep_duration_minutes"], ascending=[True, True])
    .head(5)[standout_columns]
    .copy()
)

best_nights["sleep_date"] = best_nights["sleep_date"].dt.date
worst_nights["sleep_date"] = worst_nights["sleep_date"].dt.date

best_nights


,sleep_date,sleep_score,sleep_duration_minutes,avg_stress,resting_heart_rate,hrv_value,steps,workout_count
106,2025-05-24,95,537.0,30.0,46,48.0,7314,0
111,2025-05-29,94,519.0,34.0,47,50.0,3808,0
72,2025-04-19,90,521.0,31.0,48,54.0,6407,1
15,2025-02-20,89,515.0,20.0,50,56.0,2029,1
1,2025-02-02,87,471.0,29.0,50,45.0,4452,0


In [10]:
worst_nights


,sleep_date,sleep_score,sleep_duration_minutes,avg_stress,resting_heart_rate,hrv_value,steps,workout_count
34,2025-03-11,32,386.0,42.0,58,37.0,10143,0
115,2025-06-02,36,203.0,44.0,52,43.0,6043,0
79,2025-04-26,38,244.0,45.0,50,42.0,20047,1
52,2025-03-30,39,285.0,42.0,57,41.0,17168,0
12,2025-02-17,42,211.0,33.0,52,50.0,13751,0


## Short Written Summary

This final cell turns the computed values into a concise narrative that can be reused in README updates or interview talking points.


In [11]:
strongest_positive = sleep_score_corr[sleep_score_corr["correlation_with_sleep_score"] > 0].head(1)
strongest_negative = sleep_score_corr[sleep_score_corr["correlation_with_sleep_score"] < 0].head(1)

workout_day_row = workout_summary.loc[workout_summary["day_type"] == "workout_day"]
no_workout_day_row = workout_summary.loc[workout_summary["day_type"] == "no_workout_day"]

summary_lines = [
    f"Analysis window: {START_DATE} to {END_DATE} ({len(df)} nights).",
]

if not strongest_positive.empty:
    row = strongest_positive.iloc[0]
    summary_lines.append(
        f"Strongest positive correlation with sleep_score: {row['metric']} ({row['correlation_with_sleep_score']:+.3f})."
    )

if not strongest_negative.empty:
    row = strongest_negative.iloc[0]
    summary_lines.append(
        f"Strongest negative correlation with sleep_score: {row['metric']} ({row['correlation_with_sleep_score']:+.3f})."
    )

if not workout_day_row.empty and not no_workout_day_row.empty:
    workout_score = workout_day_row["avg_sleep_score"].iloc[0]
    no_workout_score = no_workout_day_row["avg_sleep_score"].iloc[0]
    summary_lines.append(
        f"Workout days average {workout_score:.1f} sleep_score vs {no_workout_score:.1f} on non-workout days."
    )

if not best_nights.empty:
    row = best_nights.iloc[0]
    summary_lines.append(
        f"Best night: {row['sleep_date']} with sleep_score {row['sleep_score']:.0f} and {row['sleep_duration_minutes']:.0f} minutes of sleep."
    )

if not worst_nights.empty:
    row = worst_nights.iloc[0]
    summary_lines.append(
        f"Worst night: {row['sleep_date']} with sleep_score {row['sleep_score']:.0f} and avg_stress {row['avg_stress']:.0f}."
    )

print("\n".join(summary_lines))


Analysis window: 2025-02-01 to 2025-06-30 (140 nights).
Strongest positive correlation with sleep_score: hrv_value (+0.339).
Strongest negative correlation with sleep_score: avg_stress (-0.367).
Workout days average 72.3 sleep_score vs 69.1 on non-workout days.
Best night: 2025-05-24 with sleep_score 95 and 537 minutes of sleep.
Worst night: 2025-03-11 with sleep_score 32 and avg_stress 42.
